# 2 Base models: does the expensive model earn its keep?

`01_windowing` turned one transformer's history into `(N, 96)` windows of history and
`(N, 24)` futures. Now we predict.

The temptation is to reach straight for the architecture with the best story attached
— it has *memory*, it has *gates*, it has *attention*. This notebook builds the ladder
in order instead, and reports two numbers for every rung: how well it did, and what it
cost. Those two columns disagree, and the disagreement is the lesson.

Three habits get built here, and they outlive every architecture in this course:

1. **Look at the data before you pick a model.** The spectrum and the autocorrelation
   tell you which baselines will be hard to beat, before you train anything.
2. **Name your baseline.** "MASE 0.9" is not a result until you say *against what*.
3. **Run it more than once.** A single seed is an anecdote. Half the rankings people
   report in blog posts do not survive three seeds, and you are about to watch that
   happen on your own screen.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from ett import load_ett, window_index
from notebooktester import param

W, H = 96, 24          # four days of history, one day of horizon
EPOCHS = param(10, test=1)
SEEDS = (0, 1, 2)

ett = load_ett()
mu, sigma = ett.values[ett.splits["train"]].mean(), ett.values[ett.splits["train"]].std()


def windows(split: str, stride: int = 1):
    # scaled on the training statistics, split in time first - 01_windowing section 2
    values = (ett.values[ett.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train")
xva, yva = windows("val")
xte, yte = windows("test")
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")

## 2.1 Look first: what periods are actually in this series?

A model can only use structure that exists. Before choosing between architectures,
ask the data what it is made of — and the cheapest way to ask is the **power
spectrum**: decompose the series into sine waves and see which periods carry energy.

Two details matter and both are one line of code. Subtract the mean, or the constant
offset dominates every bin. Apply a **taper** (here a Hann window), or the fact that
the series does not conveniently start and end at the same value smears energy across
the whole spectrum — the standard *spectral leakage* artefact, and the reason a raw
`rfft` of a trending series looks like noise at every scale.

In [ ]:
train_values = (ett.values[ett.splits["train"]] - mu) / sigma
n = len(train_values)

z = (train_values - train_values.mean()) * torch.hann_window(n, periodic=False)
power = torch.fft.rfft(z).abs() ** 2
freqs = torch.fft.rfftfreq(n, d=1.0)          # cycles per hour
periods = 1.0 / freqs[1:]                     # hours per cycle, skipping DC

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.loglog(periods, power[1:], lw=0.6)
for hours, label in ((12, "12h"), (24, "24h"), (168, "1 week"), (24 * 365, "1 year")):
    ax.axvline(hours, color="crimson", ls="--", lw=1)
    ax.text(hours, power[1:].max(), f" {label}", color="crimson", va="top", fontsize=9)
ax.set_xlabel("period (hours)")
ax.set_ylabel("power")
ax.set_title("ETTh1 oil temperature — power spectrum of the training year")
plt.tight_layout()

Compare this with what you would expect from, say, pedestrian counts or electricity
demand, where the 24-hour spike towers over everything. Oil temperature is a
**thermal** quantity: it integrates load rather than tracking it, so most of its
energy sits at long periods and the daily cycle is a modest bump on a slope.

The autocorrelation says the same thing in units you can act on.

In [ ]:
def acf(x: torch.Tensor, max_lag: int) -> torch.Tensor:
    xc = x - x.mean()
    denom = (xc * xc).sum()
    return torch.tensor([1.0] + [float((xc[lag:] * xc[:-lag]).sum() / denom) for lag in range(1, max_lag + 1)])


a = acf(train_values, 200)
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(a, lw=1)
ax.axhline(0, color="k", lw=0.5)
for lag in (24, 48, 168):
    ax.axvline(lag, color="crimson", ls="--", lw=1)
ax.set_xlabel("lag (hours)")
ax.set_ylabel("autocorrelation")
plt.tight_layout()

for lag in (1, 24, 48, 168):
    print(f"acf({lag:3d}h) = {a[lag]:+.3f}")

`acf(1h)` is very close to 1: **the best single predictor of the next hour is this
hour**. `acf(24h)` is far weaker — yesterday at this time is not a strong guide. Write
down the prediction that follows, *before* measuring it:

> repeating the last value will be a strong baseline, and repeating yesterday's
> twenty-four hours will be a weak one.

If you are used to daily-seasonal data this feels backwards. That is exactly why you
look first. On the pedestrian or electricity series it would come out the other way,
and the same code would have told you so.

## 2.2 Which naive model? (the question that makes MASE mean something)

Two baselines, both free:

- **naive** — repeat the last observed value for all 24 steps.
- **seasonal naive** — repeat the last 24 observed values, i.e. "tomorrow is
  yesterday".

And **MASE**, which is just MAE divided by a reference error so that the number is
comparable across series of different units and sizes. The textbook reference is the
in-sample one-step naive error on the *training* data, so it does not move when you
change evaluation split — and it is one line, so we write it out rather than import it.

In [ ]:
def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


# MASE scale: mean absolute one-step change of the training series (Hyndman's definition)
mase_scale = float((train_values[1:] - train_values[:-1]).abs().mean())
print(f"MASE scale (mean |y_t - y_t-1| on train) = {mase_scale:.4f}")

naive = xva[:, -1:].expand(-1, H)
seasonal = xva[:, -24:][:, :H]

results: dict[str, dict] = {}
results["naive (repeat last)"] = {"mae": mae(yva, naive), "std": 0.0, "params": 0, "secs": 0.0}
results["seasonal naive (24h)"] = {"mae": mae(yva, seasonal), "std": 0.0, "params": 0, "secs": 0.0}

for name, r in results.items():
    print(f"{name:24s} MAE {r['mae']:.4f}   MASE {r['mae'] / mase_scale:.3f}")

The spectrum was right: the seasonal baseline is the **worse** of the two here. Which
is the whole point of naming your baseline. "Our model reaches MASE 0.9" is a claim
about the model *and* the reference; report one without the other and the number is
decoration. Anyone quoting a MASE at you should be able to say, immediately, what its
denominator is.

Both baselines cost zero parameters and zero seconds. Everything below has to earn the
difference.

## 2.3 A linear layer

`nn.Linear(96, 24)`. Every output hour is a weighted sum of the 96 input hours. No
nonlinearity, no memory, no gates — 2328 parameters that a spreadsheet could hold.

Treat this as the baseline to *beat*, not as a warm-up you scroll past. The
**LTSF-Linear** paper (Zeng et al., 2022, *Are Transformers Effective for Time Series
Forecasting?*) made its name by showing that this layer matches or beats a stack of
published transformers on most long-horizon benchmarks. That result is reproducible on
a laptop, and you are about to reproduce it.

In [ ]:
def fit(make_model, seeds=SEEDS, epochs=EPOCHS, batch_size=64, lr=1e-3):
    # trains the same architecture once per seed; returns mean/std of val MAE, params, seconds
    scores, secs = [], []
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
        t0 = time.time()
        for _ in range(epochs):
            for xb, yb in loader:
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
        secs.append(time.time() - t0)
        model.eval()
        with torch.no_grad():
            scores.append(mae(yva, model(xva)))
    return {
        "mae": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "params": sum(p.numel() for p in model.parameters()),
        "secs": float(np.mean(secs)),
        "model": model,  # the last seed, kept for plotting
    }


def report(name: str, r: dict) -> None:
    results[name] = r
    print(f"{name:24s} MAE {r['mae']:.4f} +/- {r['std']:.4f}   MASE {r['mae'] / mase_scale:.3f}   "
          f"{r['params']:>7,} params   {r['secs']:5.1f}s")


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):          # (B, W) -> (B, H)
        return self.fc(x)


report("linear", fit(Linear))

## 2.4 Convolutions over time

A `Conv1d` slides a small filter along the time axis. Before using one, get the axis
order straight, because it is the single most common source of silently-wrong
timeseries code:

- **`(B, T, F)`** — what `nn.RNN(batch_first=True)`, and this course, calls a batch of
  sequences. Time is axis 1, features axis 2.
- **`(B, C, T)`** — what `nn.Conv1d` wants. *Channels* on axis 1, time last.

These are the same numbers in a different order, and both are rank 3. `jaxtyping` will
not save you: `(32, 96, 1)` and `(32, 1, 96)` are both valid `(B, C, T)` as far as any
shape checker is concerned, and only one of them means what you meant. This is the
formal/semantic split from `tensor-ladder.md`, in its most expensive form — the wrong
one trains happily and predicts noise.

Look at what one convolution actually does to a slice of the series:

In [ ]:
torch.manual_seed(0)
conv = nn.Conv1d(in_channels=1, out_channels=5, kernel_size=3)
assert conv.bias is not None       # bias=True by default; ty needs this spelled out

segment = train_values[:336].reshape(1, 1, -1)     # (B=1, C=1, T=336) — two weeks
filtered = conv(segment).detach()

fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
ax[0].plot(segment[0, 0], lw=0.8, color="k")
ax[0].set_ylabel("input")
for c in range(5):
    ax[1].plot(filtered[0, c], lw=0.7, label=f"channel {c}")
ax[1].set_ylabel("5 filtered channels")
ax[1].set_xlabel("hours")
ax[1].legend(ncol=5, fontsize=8)
fig.suptitle("one untrained Conv1d(1, 5, 3): five different views of the same series")
plt.tight_layout()

print(f"{tuple(segment.shape)} -> {tuple(filtered.shape)}   ({conv.weight.numel()} weights + {conv.bias.numel()} biases)")

Five channels, five different local patterns picked out of the same input — and these
filters are *random*, nobody chose them. Training chooses them. That is the whole
difference between a convolution and the hand-designed filter you would write yourself
(you will write some of those in `04_composition`).

Now the question that decides whether convolutions can work here at all: **how far
back can this thing see?** A kernel of size 3 sees 3 steps. Stack another and it sees
5. The receptive field grows *linearly* with depth and *exponentially* with dilation.

In [ ]:
def receptive_field(layers: int, kernel: int = 3, dilation_growth: int = 1) -> int:
    rf, dilation = 1, 1
    for _ in range(layers):
        rf += (kernel - 1) * dilation
        dilation *= dilation_growth
    return rf


print(f"{'layers':>7} {'plain (k=3)':>13} {'dilated x2':>12}")
for layers in (1, 2, 4, 8, 12):
    print(f"{layers:>7} {receptive_field(layers):>13} {receptive_field(layers, dilation_growth=2):>12}")

print(f"\nto cover one day  ( 24h): {int(np.ceil(24 / 2)):>3} plain layers, or {4:>2} dilated")
print(f"to cover one week (168h): {int(np.ceil(168 / 2)):>3} plain layers, or {8:>2} dilated")

Eighty-four stacked layers to see a week, or eight if you double the dilation each
time. This arithmetic is why dilated convolutions (WaveNet, TCN) exist, and it is also
the honest motivation for attention: a self-attention layer reaches the whole window in
*one* step, at quadratic cost. Hold that thought until lesson 5.

Our windows are 96 steps, so a modest stack with pooling covers them. Let us train one.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, channels: int = 16):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv1d(1, channels, 3, padding=1), nn.ReLU(),
            nn.Conv1d(channels, channels, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(channels, channels, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.head = nn.Linear(channels * (W // 4), H)

    def forward(self, x):                       # (B, W) -> (B, H)
        z = self.body(x.unsqueeze(1))           # (B, 1, W) -> (B, C, W/4)   <- the (B, C, T) convention
        return self.head(z.flatten(1))


report("conv1d x3", fit(ConvNet))

## 2.5 Recurrence

An RNN walks the sequence one step at a time, carrying a hidden state. Unlike the
convolution, its receptive field is unbounded in principle — step 96 can in principle
depend on step 1. In practice the gradient has to travel back through 96 multiplications
to say so, which is the vanishing-gradient problem gates were invented to fix.

Get the output shapes straight first. `nn.RNN` returns **two** things and people reach
for the wrong one constantly:

In [ ]:
demo = nn.RNN(input_size=1, hidden_size=8, num_layers=1, batch_first=True)
out, hidden = demo(xtr[:4].unsqueeze(-1))       # (B, T, F) with F=1

print(f"input   {tuple(xtr[:4].unsqueeze(-1).shape)}   (B, T, F)")
print(f"out     {tuple(out.shape)}   the hidden state at *every* timestep")
print(f"hidden  {tuple(hidden.shape)}   the final state only, (layers, B, hidden)")
print(f"\nsame tensor? {torch.allclose(out[:, -1, :], hidden[0])}")

`out[:, -1, :]` and `hidden[-1]` are the same numbers. Use `out` when you want a
prediction per timestep (sequence labelling), the last state when you want one
prediction per sequence — which is what forecasting a fixed horizon from a fixed window
is. Slicing `out[:, -1, :]` is the more portable habit, because it survives swapping
`nn.RNN` for `nn.LSTM`, whose second return value is a *tuple* of two states.

Three cells of the same shape, three amounts of machinery:

| | what it adds | parameters per unit |
|---|---|---|
| `RNN` | nothing — one matrix and a tanh | 1x |
| `GRU` | reset and update gates | 3x |
| `LSTM` | input/forget/output gates plus a cell state | 4x |

Same interface, so one class covers all three.

In [ ]:
class Recurrent(nn.Module):
    def __init__(self, kind: str = "gru", hidden: int = 64, layers: int = 1):
        super().__init__()
        cell = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.rnn = cell(input_size=1, hidden_size=hidden, num_layers=layers, batch_first=True)
        self.head = nn.Linear(hidden, H)

    def forward(self, x):                       # (B, W) -> (B, H)
        out, _ = self.rnn(x.unsqueeze(-1))      # (B, W, 1) -> (B, W, hidden)
        return self.head(out[:, -1, :])


for kind in ("rnn", "gru", "lstm"):
    report(f"{kind.upper()}(64)", fit(lambda k=kind: Recurrent(k)))

## 2.6 The table, with the cost column

Now put them side by side. The columns nobody prints are the ones that decide whether
you would actually deploy the thing.

In [ ]:
order = sorted(results, key=lambda k: results[k]["mae"])
print(f"{'model':24s} {'val MAE':>9} {'+/-':>7} {'MASE':>6} {'vs naive':>9} {'params':>9} {'secs':>7}")
print("-" * 76)
base = results["naive (repeat last)"]["mae"]
for name in order:
    r = results[name]
    print(f"{name:24s} {r['mae']:9.4f} {r['std']:7.4f} {r['mae'] / mase_scale:6.3f} "
          f"{r['mae'] / base:9.3f} {r['params']:9,} {r['secs']:7.1f}")

Read the `+/-` column before you read the ranking, because it changes what the ranking
means. Every trained model lands in a band roughly `0.172` to `0.184`, and several of
the gaps between neighbouring rows are **smaller than the seed spread of the rows
themselves**. Those rows are not ordered. Re-run the notebook with seeds `(3, 4, 5)`
and they will change places. A comparison whose margin is inside its own noise is a
coin flip with a table around it, and most of the timeseries leaderboards you will read
are exactly this table with the `+/-` column deleted.

What *does* survive:

- **Every trained model beats both free baselines, and none of them beats them by
  much.** The whole trained field spans about 6% of MAE. That is the honest headline.
- **The cost column spans two orders of magnitude for that 6%.** The linear layer is
  2328 parameters and a fraction of a second; the LSTM is eight times the parameters
  and roughly a hundred times the seconds, for a score inside the noise band. The
  LTSF-Linear paper (Zeng et al., 2022) built its case on this observation, and here it
  is on your laptop.
- **The seed spreads themselves differ by an order of magnitude.** `linear` reproduces
  to four decimals; the deeper models wobble. That column is not noise about the
  measurement, it is information about the model — and section 2.7 is about to show you
  what it predicts.

None of this says recurrence is useless — it says recurrence is not useful **on this
task**. `06_exercise` runs the same family on sequence *classification*, where the
answer flips. A rule you can carry out of here: reach for memory when the thing you
need to remember is far away and non-local. Predicting the next few hours of a smooth
thermal series with `acf(1)` near 1 is neither — most of the answer is the last value,
and a linear layer has a direct path to it by construction.

Look at what the two ends of the table actually predict:

In [ ]:
picks = [0, 700, 1400, 2100]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 3), sharey=True)
with torch.no_grad():
    curves = {name: results[name]["model"](xva[picks]) for name in ("linear", "GRU(64)") if "model" in results[name]}

for ax, i in zip(axes, range(len(picks))):
    ax.plot(range(W), xva[picks][i], color="0.6", lw=1, label="history")
    ax.plot(range(W, W + H), yva[picks][i], color="k", lw=1.5, label="truth")
    for name, c in curves.items():
        ax.plot(range(W, W + H), c[i], lw=1.2, ls="--", label=name)
    ax.axvline(W, color="0.8", lw=1)
axes[0].set_ylabel("scaled OT")
axes[-1].legend(fontsize=8)
fig.suptitle("four validation windows: 96 hours in, 24 hours out")
plt.tight_layout()

Both models mostly continue the last value and bend it a little. That is not a failure
of either — it is what the data supports at this horizon. A forecast that looks boring
on a series with `acf(1) ≈ 0.99` is a forecast that is paying attention.

It also exposes the real limitation of everything in this notebook: **a single line is
a claim of certainty that none of these models are entitled to make.** The next
notebook fixes that.

## 2.7 The test split, once

Every number above is validation. We chose `W`, `H`, the epoch count, the hidden sizes
and the ranking by looking at those numbers, which means they are now partly a
description of the validation split. The test months exist to find out how much.

In [ ]:
print(f"{'model':24s} {'val MAE':>9} {'val +/-':>8} {'test MAE':>9} {'val rank':>9} {'test rank':>10}")
print("-" * 74)
test_naive = mae(yte, xte[:, -1:].expand(-1, H))

trained = [n for n in order if "model" in results[n]]
with torch.no_grad():
    test_scores = {n: mae(yte, results[n]["model"](xte)) for n in trained}
test_order = sorted(trained, key=lambda n: test_scores[n])

print(f"{'naive (repeat last)':24s} {base:9.4f} {'-':>8} {test_naive:9.4f}")
for name in trained:
    r = results[name]
    print(f"{name:24s} {r['mae']:9.4f} {r['std']:8.4f} {test_scores[name]:9.4f} "
          f"{trained.index(name) + 1:>9} {test_order.index(name) + 1:>10}")

Three things to notice, and none of them is comfortable.

**The test errors are lower than the validation errors.** The last four months are
simply an easier stretch of this series than the four before them. A test score
measures a model *and* the period you measured it on, so "our error dropped" can mean
nothing more than "winter ended". Never compare a score across splits and call the
difference progress.

**The ranking is not the same.** The two rank columns disagree, and not by a little.
The model that came top on validation is not the model that comes top here. Choosing
`W`, `H`, the epoch count, the hidden sizes and finally the winner by looking at
validation numbers means those numbers now partly describe the validation split rather
than the problem. This is the same result `5_attention/05_long_documents` reaches from
the other direction: the best trial of a search is not the best configuration.

**The seed spread predicted it.** Look at the `val +/-` column next to the size of each
model's val-to-test move. The model that reproduced most tightly across seeds also
travels most stably across time; the ones that wobbled between seeds wobble between
periods too. Seed variance is not a nuisance to average away — it is the cheapest
available estimate of how much of a score is the model and how much is the particular
run. You get it for the price of a `for` loop.

So: is the small trained gain over `naive` real? On this evidence, partly — the
direction survives, the ordering inside the field does not. That answer is less
satisfying than a leaderboard and considerably more likely to still be true next month.

## What to take from this

- Read the spectrum and the autocorrelation before choosing an architecture. They cost
  two cells and they told us in advance which baseline would be strong.
- Report the baseline your metric is scaled against, always.
- Report seconds and parameters next to the score, always. A tie at 80x the cost is a
  loss.
- Run three seeds. Rankings inside the noise band are not rankings.
- A point forecast cannot express uncertainty, and on this data uncertainty is most of
  what there is to say. That is `03_probabilistic`.

## Exercises

**E1 — window length.** `W=96` was asserted, not derived. Re-run the linear model at
`W` of 24, 48, 96, 192 and 336. Where does it stop helping, and does the answer match
what the spectrum in 2.1 predicts?

**E2 — horizon.** Keep `W=96` and push `H` to 48, 96, 192. Plot MASE against horizon
for the linear model and the GRU. Does the gap between them change sign anywhere? (The
LTSF-Linear paper's central claim is about exactly this curve.)

**E3 — stride.** `01_windowing` showed consecutive windows overlap by 95/96. Re-run the
GRU with `windows("train", stride=6)` and `stride=24`. Plot score against training
seconds. How much data can you throw away before it hurts?

**E4 — the covariates.** `ett.covariates` holds six load channels this notebook ignored
entirely. Feed them to the recurrent model by stacking them onto the input feature axis
(`input_size=7` instead of `1`). Oil temperature is *caused* by load — does telling the
model so help, and does it help the linear model too?

**E5 — a harder series.** Swap ETTh1 for `Monash-University/monash_tsf` config
`pedestrian_counts` (66 sensors, hourly, and a daily cycle that dominates its spectrum).
Re-run 2.1 first. Which baseline is strong there, and does the ranking of the trained
models change?